Step 1: Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import string
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud
nltk.download("stopwords")

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

Step 2: Load the Dataset

In [ ]:
data = pd.read_csv('data/spam_ham_dataset.csv')
data.head()

In [ ]:
data.shape

In [ ]:
sns.countplot(x='label', data=data)
plt.show()

Step 3: Balancing Ham and Spam Email Dataset

In [ ]:
ham_msg = data[data['label'] == 'ham']
spam_msg = data[data['label'] == 'spam']

# Balancing the dataset by sampling the ham messages to match the number of spam messages
ham_msg_balanced = ham_msg.sample(spam_msg.shape[0], random_state=42) #42 is used for reproducibility

# Concatenating the balanced ham messages with the spam messages
balanced_data = pd.concat([ham_msg_balanced, spam_msg], axis=0).reset_index(drop=True)
balanced_data.shape

sns.countplot(x='label', data=balanced_data)
plt.title('Balanced Dataset Distribution of Spam and Ham Messages')
plt.xticks(ticks=[0, 1], labels=['Ham', 'Spam'])
plt.show()

Step 4: Clean the Text - We do not neeed the text written Subject so we can clear that out

In [ ]:
balanced_data['text'] = balanced_data['text'].str.replace('Subject', '')
balanced_data.head()

In [ ]:
# Removing punctuations from the text
punctuations_list = string.punctuation
def remove_punctuations(text):
    temp = str.maketrans('', '', punctuations_list)
    return text.translate(temp)

balanced_data['text'] = balanced_data['text'].apply(lambda x: remove_punctuations(x))
balanced_data.head()

In [ ]:
# Removing stopwords from the text
def remove_stopwords(text):
    stop_words = set(stopwords.words('english'))
    imp_words = []

    for word in str(text).split():
        word = word.lower()

        if word not in stop_words:
            imp_words.append(word)
    
    output = ' '.join(imp_words)
    return output

balanced_data['text'] = balanced_data['text'].apply(lambda x: remove_stopwords(x))
balanced_data.head()

Step 5: Visualising Word Cloud

In [ ]:
def plot_wordcloud(data, typ):
    email_corpus = ' '.join(data['text'])
    worldcloud = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(email_corpus)
    plt.figure(figsize=(7, 7))
    plt.imshow(worldcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for {typ} Emails', fontsize=15)
    plt.show()

plot_wordcloud(balanced_data[balanced_data['label'] == 'ham'], typ = 'Non-spam')
plot_wordcloud(balanced_data[balanced_data['label'] == 'spam'], typ = 'Spam')

Step 6: Tokenisation and Padding

In [ ]:
max_words = 10000
max_len = 100

train_X, test_X, train_Y, test_Y = train_test_split(
    balanced_data['text'],
    balanced_data['label'],
    test_size=0.2,
    random_state=42,
    stratify=balanced_data['label']
)
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_X)

train_sequences = tokenizer.texts_to_sequences(train_X)
test_sequences = tokenizer.texts_to_sequences(test_X)

train_sequences = pad_sequences(
    train_sequences,
    maxlen=max_len,
    padding='post',
    truncating='post'
)

test_sequences = pad_sequences(
    test_sequences,
    maxlen=max_len,
    padding='post',
    truncating='post'
)

train_Y = (train_Y == 'spam').astype(int)
test_Y = (test_Y == 'spam').astype(int)

Step 7: Define the Model

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.Input(shape=(max_len,)),

    tf.keras.layers.Embedding(
        input_dim=max_words,
        output_dim=32
    ),

    tf.keras.layers.LSTM(16),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Step 8: Train the Model

In [ ]:
es = EarlyStopping(patience=3, monitor='val_accuracy', restore_best_weights=True)
lr = ReduceLROnPlateau(patience=2, monitor='val_loss', factor=0.5, verbose=0)

history = model.fit(
    train_sequences, train_Y,
    validation_data=(test_sequences, test_Y),
    epochs=20,
    batch_size=32,
    callbacks=[lr, es]
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_sequences, test_Y)
print('Test Loss :',test_loss)
print('Test Accuracy :',test_accuracy)

In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

In [ ]:
def clean_new_email(email_text):
    """
    Clean a new email using the same cleaning steps used for the training data.
    """

    # Convert email to string
    email_text = str(email_text)

    # Remove the word "Subject" like we did in the dataset
    email_text = email_text.replace("Subject", "")
    email_text = email_text.replace("subject", "")

    # Remove punctuation
    email_text = remove_punctuations(email_text)

    # Remove stopwords
    email_text = remove_stopwords(email_text)

    return email_text


def predict_email(email_text):
    """
    Predict whether a new email is spam or ham.
    """

    # Step 1: Clean the email
    cleaned_email = clean_new_email(email_text)

    # Step 2: Convert the email text into numbers using the SAME tokenizer
    email_sequence = tokenizer.texts_to_sequences([cleaned_email])

    # Step 3: Pad the sequence using the SAME max_len
    email_padded = pad_sequences(
        email_sequence,
        maxlen=max_len,
        padding='post',
        truncating='post'
    )

    # Step 4: Predict using the trained model
    spam_probability = model.predict(email_padded, verbose=0)[0][0]

    # Step 5: Convert probability into label
    if spam_probability >= 0.5:
        prediction = "Spam"
    else:
        prediction = "Ham"

    return {
        "Original Email": email_text,
        "Cleaned Email": cleaned_email,
        "Spam Probability": float(spam_probability),
        "Prediction": prediction
    }

In [ ]:
my_email = """
##Add your email text here to test the model. For example:
Subject: Congratulations! You have won a free iPhone.
Click this link now to claim your prize.
"""

result = predict_email(my_email)
result